# 교안 02. 표준 ID를 연결하고 새 관계를 증분 적재합니다

**교안 01에서 기존 관계를 DB에 준비하고, 새 관계만 추출해 파일로 저장했습니다.**  
이 교안은 그 파일의 이름을 표준 ID에 연결하고, 문서와 청크를 중복 없이 저장한 뒤 새 관계를 추가합니다.

<img src="./images/document_chunk_reuse.png" width="1000" alt="같은 doc_id의 문서와 고정 chunk_id의 청크를 재사용하며, 표준 개체에서 출처 청크와 문서로 연결합니다. 새 관계를 다시 저장해도 문서와 청크, 관계는 늘지 않습니다.">

| 자료 | 이미 저장된 관계 | 이번 추가 |
|---|---|---|
| 영화 | 출연 8건 | 장르 1건 |
| 의료 | 치료와 완화 4건 | 증상 완화 6건 |

위 추가 수는 원문 기준으로 확인할 수 있는 관계 수입니다. 실제 추출 결과는 원문과 대조합니다.

**실습의 목표**

1. 새 관계의 양 끝을 기존 ID 또는 검토한 새 ID에 연결합니다.
2. 문서 ID와 청크 키로 원문을 재사용하고, 개체와 관계의 중복 저장을 막습니다.
3. 새 관계를 추가하고, 기존 관계 보존과 재실행 결과를 확인합니다.
4. 서로 다른 개념을 합치지 않고 계층으로 연결해 조회합니다.
5. 적재 이력을 기록하고 이번에 추가한 관계만 되돌립니다.

교안 01의 본문 또는 핵심 코드를 먼저 실행하세요. 여기서는 기존 관계를 다시 추출하거나 초기 적재하지 않습니다.

#### 라이브러리와 Neo4j 연결

`.env`의 접속 정보로 연결하고, `run_cypher`로 쿼리를 실행합니다. 데이터와 결과 파일의 경로도 준비합니다.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from pprint import pprint
from uuid import uuid4
from urllib.parse import urlsplit

from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 리스트로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)

## 1. 새 관계의 이름을 표준 ID에 연결합니다

**기존 개체의 ID는 재사용하고, 새 개체만 등록합니다.** 빌더의 임시 노드 `id`는 프로젝트의 `standard_id`가 아닙니다.

| 원문으로 확인한 대상 | 처리 |
|---|---|
| 기존 개체의 이름 또는 검토된 별칭 | 기존 ID 연결 |
| 목록에 없는 새 개체 | 검토한 새 ID 등록 |
| 기존 개체보다 넓거나 좁은 개념 | 별도 ID 유지, 필요하면 계층 연결 |
| 대상을 확정할 수 없음 | ID 보류 유지 |

ID를 붙여도 잘못된 인용이나 관계의 의미가 교정되지는 않습니다.

### 1-1. 영화의 새 장르 관계에 ID 연결

`The Devil's Advocate`는 이미 있는 영화입니다. 새 장르 `supernatural horror`는 더 넓은 `horror`와 다른 ID를 사용합니다.

In [ ]:
# demo_extraction_packet.json: 앞 교안에서 만든 새 관계, 원문과 검사 결과입니다.
demo_packet = json.loads(
    (output_dir / "demo_extraction_packet.json").read_text(encoding="utf-8")
)
demo_batch = demo_packet["rows"]  # 검사 전 새 추출 전체입니다.
demo_validated = demo_packet["validated"]  # 두 검사를 통과해 ID를 연결할 행입니다.
demo_rejected = demo_packet["rejected"]  # 교안 01의 기각 사유를 그대로 보존합니다.
demo_docs = demo_packet["documents"]
demo_signatures = demo_packet["signatures"]
demo_schema_version = demo_packet["schema_version"]

# 기존 그래프의 개체 목록입니다. 이 셀에서 기존 관계를 다시 저장하지 않습니다.
demo_existing = read_json(demo_packet["baseline_file"])
demo_catalog = demo_existing["catalog"]
demo_baseline = demo_existing["rows"]
print("파일에서 받은 새 관계:", len(demo_batch))
print("검사 통과:", len(demo_validated), "/ 검사 기각:", len(demo_rejected))
for row in demo_validated:
    print("ID를 연결할 관계:", (row["subject"], row["relation"], row["object"]))

#### 검토한 새 장르를 개체 목록에 추가하기

기존 ID는 유지하고, 원문으로 확인한 새 장르의 ID를 목록에 더합니다. 아직 DB에 저장하는 단계는 아닙니다.

In [ ]:
# demo_new_entities.json: 원문으로 확인한 새 개체의 이름, 별칭과 내부 표준 ID입니다.
demo_approved = read_json("demo_new_entities.json")
demo_expanded_catalog = demo_catalog + demo_approved
for entity in demo_approved:
    print("이름:", entity["name"], "/ 표준 ID:", entity["standard_id"])
    print("확인한 내용:", entity["review_note"])
print("기존 개체:", len(demo_catalog), "/ 보완한 목록:", len(demo_expanded_catalog))

#### 타입과 이름으로 ID를 찾는 딕셔너리 만들기

`demo_ids_by_name`은 `(타입, 별칭)`을 후보 ID 집합에 연결합니다. 예를 들어 `('Movie', "The Devil's Advocate")`로 기존 영화 ID를 찾습니다.

In [ ]:
# (타입, 별칭)을 키로 사용합니다. 이름이 같아도 타입이 다르면 다른 키입니다.
demo_ids_by_name = {}
for entity in demo_expanded_catalog:
    for alias in entity["aliases"]:
        key = (entity["entity_type"], alias)
        # 같은 ID가 중복 등록되어도 후보 하나로 세도록 집합에 담습니다.
        demo_ids_by_name.setdefault(key, set()).add(entity["standard_id"])

# 첫 관계의 두 이름으로 조회 결과를 확인합니다.
for role in ("subject", "object"):
    row = demo_validated[0]
    key = (row[role + "_type"], row[role])
    print("타입과 이름:", key, "/ 후보 ID:", sorted(demo_ids_by_name.get(key, set())))

#### 두 이름에 ID를 붙이고 적재 가능과 보류 구분하기

`demo_validated`의 각 행을 복사해 `subject_id`, `object_id`를 붙입니다. 후보가 하나로 정해지지 않는 이름은 `unresolved_names`에 남겨 보류합니다.

In [ ]:
demo_ready, demo_pending = [], []
# 교안 01의 검사 통과 행에만 ID를 연결합니다. 원본 대신 복사본에 기록합니다.
for original in demo_validated:
    row = dict(original)
    unresolved = []
    for role in ("subject", "object"):
        key = (row[role + "_type"], row[role])
        candidates = demo_ids_by_name.get(key, set())
        if len(candidates) == 1:
            row[role + "_id"] = next(iter(candidates))  # 하나뿐인 ID를 꺼냅니다.
        else:
            unresolved.append(row[role])  # 후보가 없거나 여러 개이면 보류합니다.

    if unresolved:
        demo_pending.append(dict(row, unresolved_names=unresolved))
    else:
        demo_ready.append(row)

print("적재 가능:", len(demo_ready), "/ ID 보류:", len(demo_pending), "/ 검사 기각:", len(demo_rejected))
for row in demo_ready:
    print("표준 ID 연결:", row["subject_id"], "->", row["relation"], "->", row["object_id"])
for row in demo_pending:
    print("ID를 확인할 이름:", row["unresolved_names"])

### 🖐️ 함께 따라하기: 의료의 새 증상 ID 연결

기존 약물 ID를 유지하고, 새로 확인한 증상 세 개를 등록합니다. 치료 관계는 입력 파일에 다시 넣지 않습니다.

In [ ]:
# [제공코드]

# paper_extraction_packet.json: 앞 교안에서 만든 새 관계, 원문과 검사 결과입니다.
paper_packet = json.loads(
    (output_dir / "paper_extraction_packet.json").read_text(encoding="utf-8")
)
paper_batch = paper_packet["rows"]  # 검사 전 새 추출 전체입니다.
paper_validated = paper_packet["validated"]  # 두 검사를 통과해 ID를 연결할 행입니다.
paper_rejected = paper_packet["rejected"]  # 교안 01의 기각 사유를 그대로 보존합니다.
paper_docs = paper_packet["documents"]
paper_signatures = paper_packet["signatures"]
paper_schema_version = paper_packet["schema_version"]

# 기존 그래프의 개체 목록입니다. 이 셀에서 기존 관계를 다시 저장하지 않습니다.
paper_existing = read_json(paper_packet["baseline_file"])
paper_catalog = paper_existing["catalog"]
paper_baseline = paper_existing["rows"]
print("파일에서 받은 새 관계:", len(paper_batch))
print("검사 통과:", len(paper_validated), "/ 검사 기각:", len(paper_rejected))
for row in paper_validated:
    print("ID를 연결할 관계:", (row["subject"], row["relation"], row["object"]))

#### 검토한 증상 목록 추가하기

`paper_new_entities.json`을 `paper_approved`에 읽고 기존 `paper_catalog`에 더해 `paper_expanded_catalog`를 만드세요.

In [ ]:
# (1) paper_new_entities.json을 paper_approved에 읽고 review_note를 확인하세요.
# (2) paper_catalog + paper_approved를 paper_expanded_catalog에 담으세요.
# (3) 새 개체의 이름과 ID, 보완한 목록의 개수를 출력하세요.
# 여기에 코드를 작성하세요.

#### 의료의 타입과 이름별 ID 조회 준비

각 개체의 별칭으로 같은 ID를 찾도록 `paper_ids_by_name`을 만드세요. 후보가 여러 개여도 모두 보존합니다.

In [ ]:
# (1) paper_ids_by_name을 빈 딕셔너리로 만드세요.
# (2) expanded_catalog의 각 개체와 그 aliases를 순회하세요.
# (3) (entity_type, alias)를 키로 ID를 집합에 모으세요. 같은 ID는 한 번만 셉니다.
# (4) 첫 통과 관계의 주어와 목적어로 조회한 후보 ID를 출력하세요.
# 여기에 코드를 작성하세요.

#### 의료 관계의 두 이름에 표준 ID 붙이기

`paper_validated`에만 ID를 붙여 `paper_ready`, `paper_pending`으로 나누세요. 교안 01의 `paper_rejected`는 그대로 유지합니다.

In [ ]:
# (1) paper_ready와 paper_pending을 빈 리스트로 만드세요.
# (2) paper_validated의 행을 복사하고, subject와 object 각각의 타입과 이름으로 ID를 찾으세요.
# (3) 후보가 하나면 subject_id 또는 object_id에 기록하고, 없거나 여러 개면 이름을 unresolved에 모으세요.
# (4) 미확정 이름이 있으면 unresolved_names를 붙여 pending에, 없으면 ready에 담으세요.
# (5) 세 건수와 양 끝의 표준 ID, 보류한 이름을 출력하세요. rejected는 앞 교안의 결과를 유지합니다.
# 여기에 코드를 작성하세요.

## 2. 문서와 청크를 재사용하고 적재할 노드를 준비합니다

| 식별자 | 무엇을 구분하나요? |
|---|---|
| `standard_id` | 같은 영화, 약물, 증상 등의 개체 |
| `doc_id` | 같은 출처 문서. 관계의 `source_doc_id`와 같은 값 |
| `chunk_id` | 같은 문서의 특정 순번과 원문을 가진 청크 |
| `claim_id` | 한 문서에서 나온 주어 ID + 관계 + 목적어 ID |
| `batch_id` | 이번 실행에서 새로 저장한 추출 관계. 되돌릴 때 이 관계들만 삭제 |

<img src="./images/document_relation_keys.png" width="1000" alt="같은 문서에서 같은 관계를 다시 저장하면 기록 하나를 재사용하고 다른 문서가 같은 관계를 보고하면 출처별 기록을 남깁니다.">

**같은 문서의 같은 관계는 재사용하고, 다른 문서의 근거는 별도 기록으로 보존합니다.**  
고유성 제약은 같은 타입의 표준 ID 중복을 막습니다. 관계와 양 끝 타입은 앞의 검사 코드로 확인합니다.

**교안 01의 파일에 있던 문서와 청크를 이번에 DB에도 저장합니다.**  
처음 실행하면 생성하고, 같은 입력으로 다시 실행하면 재사용합니다. 문서 제목이 비슷하다고 같은 문서로 합치지 않습니다.

- **문서:** `doc_id`가 같으면 같은 `Document` 노드입니다. 제목과 URL을 보관합니다.
- **청크:** 문서 ID, 청크 순번과 청크 원문으로 고정 키를 만듭니다. 원문과 768차원 임베딩을 보관합니다.
- **청크 원문이 바뀐 경우:** 다른 키로 저장합니다. 변경된 문서의 이전 추출 관계를 교체하는 정책은 별도입니다.

해시는 같은 입력에서 같은 문자열을 만드는 계산입니다. 빌더가 실행마다 만드는 임시 UUID는 재사용 키로 쓰지 않습니다.  
`MERGE`와 고유성 제약으로 같은 키의 노드가 늘지 않게 합니다. [Neo4j 공식 문서](https://neo4j.com/docs/cypher-manual/current/clauses/merge/)

### 2-1. 영화 문서와 청크를 중복 없이 저장하기

문서 정보부터 준비하고, 빌더 결과의 청크를 고정 키로 변환해 저장합니다. 새 임베딩은 만들지 않습니다.

#### 저장할 영화 문서 준비

`builder_runs`에서 이번에 추출한 원문의 문서 ID, 제목과 URL을 꺼냅니다.

In [ ]:
# builder_runs에는 교안 01에서 추출한 문서와 그래프가 함께 들어 있습니다.
demo_document_rows = []
for run in demo_packet["builder_runs"]:
    document = run["document"]
    demo_document_rows.append(
        {
            "doc_id": document["doc_id"],
            "title": document["title"],
            "url": document.get("url", ""),
        }
    )
for document in demo_document_rows:
    print("문서 ID:", document["doc_id"], "/ 제목:", document["title"])

#### 청크에 재사용할 고정 ID 붙이기

**같은 문서, 같은 순번, 같은 청크 원문이면 같은 ID**를 만듭니다. 긴 원문은 `sha256` 해시로 바꿔 ID에 넣습니다.

`demo_chunk_rows`는 저장할 청크 목록, `demo_chunk_id_map`은 빌더의 임시 ID에서 고정 ID를 찾는 딕셔너리입니다.

In [ ]:
demo_chunk_rows = []
demo_chunk_id_map = {}  # 빌더의 임시 청크 ID를 DB에서 재사용할 고정 ID에 연결합니다.
for run in demo_packet["builder_runs"]:
    doc_id = run["document"]["doc_id"]
    for node in run["graph"]["nodes"]:
        if node["label"] != "Chunk":
            continue
        chunk = node["properties"]
        # 문서와 순번이 같아도 청크 원문이 바뀌면 다른 ID가 됩니다.
        text_hash = hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest()
        chunk_id = f"{doc_id}:chunk:{chunk['index']}:{text_hash}"
        demo_chunk_id_map[node["id"]] = chunk_id
        demo_chunk_rows.append(
            {
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "index": chunk["index"],
                "text": chunk["text"],
                "embedding": node["embedding_properties"]["embedding"],
                "embedding_model": run["settings"]["embedding_model"],
            }
        )

#### 저장할 청크 원문과 임베딩 확인하기

원문과 벡터는 교안 01의 추출 결과에서 가져옵니다.

In [ ]:
for chunk in demo_chunk_rows:
    print("문서:", chunk["doc_id"], "/ 청크 순번:", chunk["index"])
    print("원문:", chunk["text"])
    print("임베딩 차원:", len(chunk["embedding"]))
    print()

#### 문서 ID로 기존 노드 찾거나 만들기

제목이나 URL이 아닌 `doc_id`로 `MERGE`합니다. 아래 쿼리는 3절의 재실행에서도 그대로 사용합니다.

In [ ]:
# 같은 문서 ID를 가진 Document가 두 개 생기지 않도록 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT IF NOT EXISTS FOR (d:Document) REQUIRE d.doc_id IS UNIQUE")
demo_document_query = """
UNWIND $rows AS item
// 기존 문서가 있으면 찾고, 없을 때만 만듭니다.
MERGE (d:Document {doc_id: item.doc_id})
ON CREATE SET d.title = item.title, d.url = item.url
RETURN d.doc_id AS doc_id, d.title AS title // 찾거나 만든 문서입니다.
ORDER BY doc_id
"""
demo_stored_documents = run_cypher(demo_document_query, rows=demo_document_rows)
print("저장하거나 재사용한 문서:", demo_stored_documents)

#### 청크를 저장하고 출처 문서에 연결하기

`Chunk -[:FROM_DOCUMENT]-> Document`를 만듭니다. 같은 청크가 있으면 원문과 임베딩을 유지합니다.

In [ ]:
run_cypher("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE")
demo_chunk_query = """
UNWIND $rows AS item
MATCH (d:Document {doc_id: item.doc_id})
MERGE (c:Chunk {chunk_id: item.chunk_id})
// 처음 저장할 때만 원문과 기존 임베딩을 기록합니다. 임베딩 API를 다시 호출하지 않습니다.
ON CREATE SET c.text = item.text, c.index = item.index,
              c.embedding = item.embedding, c.embedding_model = item.embedding_model
MERGE (c)-[:FROM_DOCUMENT]->(d)
RETURN c.chunk_id AS chunk_id, c.index AS index, // 청크의 고정 ID와 순번입니다.
       size(c.embedding) AS dimensions // 저장한 벡터의 길이입니다.
ORDER BY chunk_id
"""
demo_stored_chunks = run_cypher(demo_chunk_query, rows=demo_chunk_rows)
print("저장하거나 재사용한 청크:", len(demo_stored_chunks))
for chunk in demo_stored_chunks:
    print("순번:", chunk["index"], "/ 임베딩 차원:", chunk["dimensions"])

### 🖐️ 함께 따라하기: 의료 문서와 청크를 재사용합니다

영화와 같은 키 규칙으로 저장하세요. 원문과 벡터는 교안 01의 결과를 그대로 사용합니다.

#### 의료 문서 정보와 고정 청크 ID 준비

문서 1편과 그 청크의 원문을 확인하세요.

In [ ]:
# [제공코드]

# builder_runs에는 교안 01에서 추출한 문서와 그래프가 함께 들어 있습니다.
paper_document_rows = []
for run in paper_packet["builder_runs"]:
    document = run["document"]
    paper_document_rows.append(
        {
            "doc_id": document["doc_id"],
            "title": document["title"],
            "url": document.get("url", ""),
        }
    )
for document in paper_document_rows:
    print("문서 ID:", document["doc_id"], "/ 제목:", document["title"])

#### 의료 청크의 재사용 키 준비

이 키는 재실행해도 같으며, 임시 ID와의 대응은 출처 연결에 사용합니다.

In [ ]:
# [제공코드]

paper_chunk_rows = []
paper_chunk_id_map = {}  # 빌더의 임시 청크 ID를 DB에서 재사용할 고정 ID에 연결합니다.
for run in paper_packet["builder_runs"]:
    doc_id = run["document"]["doc_id"]
    for node in run["graph"]["nodes"]:
        if node["label"] != "Chunk":
            continue
        chunk = node["properties"]
        # 문서와 순번이 같아도 청크 원문이 바뀌면 다른 ID가 됩니다.
        text_hash = hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest()
        chunk_id = f"{doc_id}:chunk:{chunk['index']}:{text_hash}"
        paper_chunk_id_map[node["id"]] = chunk_id
        paper_chunk_rows.append(
            {
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "index": chunk["index"],
                "text": chunk["text"],
                "embedding": node["embedding_properties"]["embedding"],
                "embedding_model": run["settings"]["embedding_model"],
            }
        )

#### 의료 청크 원문과 임베딩 확인하기

In [ ]:
# [제공코드]

for chunk in paper_chunk_rows:
    print("문서:", chunk["doc_id"], "/ 청크 순번:", chunk["index"])
    print("원문:", chunk["text"])
    print("임베딩 차원:", len(chunk["embedding"]))
    print()

#### 의료 문서를 중복 없이 저장하기

`paper_document_query`에 문서 저장 쿼리를 작성하고 `paper_document_rows`를 전달하세요.

In [ ]:
# (1) Document.doc_id의 고유성 제약을 만드세요.
# (2) paper_document_query에서 UNWIND $rows로 행을 꺼내고 doc_id로 MERGE하세요.
# (3) ON CREATE SET으로 title, url을 기록하고 doc_id, title을 반환하세요.
# (4) 쿼리를 실행해 paper_stored_documents에 결과를 담고 출력하세요.
# 여기에 코드를 작성하세요.

#### 의료 청크와 임베딩 저장하기

`paper_chunk_query`에 고정 ID로 청크를 저장하고 문서에 연결하는 쿼리를 작성하세요.

In [ ]:
# (1) Chunk.chunk_id의 고유성 제약을 만드세요.
# (2) UNWIND $rows에서 doc_id로 문서를 찾고 chunk_id로 청크를 MERGE하세요.
# (3) 처음 생성한 청크에 text, index, embedding, embedding_model을 저장하세요.
# (4) FROM_DOCUMENT도 MERGE하고 chunk_id, index, size(c.embedding) AS dimensions를 반환하세요.
# (5) paper_chunk_rows를 넘긴 결과를 paper_stored_chunks에 담고 건수와 차원을 출력하세요.
# 여기에 코드를 작성하세요.

### 2-2. 개체와 추출 관계를 저장할 함수 준비

원문 저장과 별도로, 앞에서 확정한 표준 ID와 관계 키를 사용합니다.

#### 새 개체의 표준 노드 준비

`entity_type`을 레이블로, `standard_id`를 식별자로 사용합니다. 같은 ID가 있으면 기존 노드를 사용합니다.

In [ ]:
def put_nodes(catalog):
    """개체 목록의 타입과 표준 ID로 노드를 찾거나 만듭니다.

    Args:
        catalog: standard_id, entity_type, name, aliases를 담은 개체 목록.
    Returns:
        처리한 개체 수. 이미 있던 노드도 포함합니다.
    """
    rows = run_cypher(
        """
    UNWIND $catalog AS item
    // 원래 타입을 레이블로 사용하고 표준 ID로 같은 개체를 찾습니다.
    MERGE (n:$(item.entity_type) {standard_id: item.standard_id})
    // 기존 이름과 별칭은 덮어쓰지 않습니다.
    ON CREATE SET n.name = item.name, n.aliases = item.aliases
    RETURN count(n) AS processed // 새로 만든 수가 아닌 처리한 개체 수입니다.
    """,
        catalog=catalog,
    )
    return rows[0]["processed"]

#### 출처를 포함한 관계 키 만들기

문서 ID, 주어 ID, 관계, 목적어 ID를 같은 순서로 해시해 `claim_id`를 만듭니다. 근거 문구를 바꿔도 이 네 값이 같으면 같은 키입니다.

In [ ]:
def claim_key(row):
    """문서 ID와 표준 ID로 표현한 관계를 저장용 키로 바꿉니다.

    Args:
        row: source_doc_id, subject_id, relation, object_id가 있는 기록.
    Returns:
        같은 문서의 같은 관계이면 동일한 문자열 ID.
    """
    values = [
        row["source_doc_id"],
        row["subject_id"],
        row["relation"],
        row["object_id"],
    ]
    # 네 값을 같은 순서로 문자열로 만든 뒤, 같은 입력에 같은 해시값을 얻습니다.
    text = json.dumps(values, ensure_ascii=False)
    return "claim:" + hashlib.sha256(text.encode("utf-8")).hexdigest()

#### 새 관계를 중복 없이 저장하기

`MERGE`로 같은 관계 키를 찾습니다. `ON CREATE SET`은 처음 만든 관계에만 근거와 묶음 ID를 써서 기존 기록을 유지합니다. 반환값은 신규 생성 수가 아닌 처리 수입니다.

In [ ]:
def put_claims(rows, batch_id, schema_version):
    """기존 노드 사이에 문서별 관계와 근거를 중복 없이 저장합니다.

    Args:
        rows: 양 끝 표준 ID를 붙인 추출 기록.
        batch_id: 이번 적재 묶음을 구분하는 ID.
        schema_version: 검사에 사용한 스키마 버전 번호.
    Returns:
        양 끝 노드를 찾아 처리한 기록 수. 기존 관계를 찾은 경우도 셉니다.
    """
    records = []
    for row in rows:
        records.append(dict(row, claim_id=claim_key(row)))
    result = run_cypher(
        """
    UNWIND $rows AS item
    // 먼저 저장해 둔 노드를 타입과 표준 ID로 찾습니다.
    MATCH (s:$(item.subject_type) {standard_id: item.subject_id})
    MATCH (o:$(item.object_type) {standard_id: item.object_id})
    // 같은 문서의 같은 관계이면 기존 관계를 재사용합니다.
    MERGE (s)-[r:$(item.relation) {claim_id: item.claim_id}]->(o)
    // 새 관계에만 출처와 근거를 기록해 이전 문서의 근거를 보존합니다.
    ON CREATE SET r.source_doc_id = item.source_doc_id,
                  r.evidence = item.evidence, r.batch_id = $batch_id,
                  r.schema_version = $schema_version
    RETURN count(r) AS processed // 이미 저장되어 있던 관계도 처리 수에 포함합니다.
    """,
        rows=records,
        batch_id=batch_id,
        schema_version=schema_version,
    )
    return result[0]["processed"]

#### 기존 관계와 새 관계 함께 조회하기

정해 둔 문서와 표준 ID 범위의 관계를 `claim_id` 순서로 읽습니다. 출처별 기록과 근거를 대조할 때 사용합니다.

In [ ]:
def read_claims(document_ids, known_ids):
    """지정한 문서들에서 내가 등록한 개체 사이의 관계와 근거를 읽습니다.

    Args:
        document_ids: 조회할 출처 문서 ID 목록.
        known_ids: 조회에 포함할 표준 ID 목록.
    Returns:
        관계와 출처를 담은 딕셔너리의 리스트. claim_id 순서로 정렬합니다.
    """
    return run_cypher(
        """
    MATCH (s)-[r]->(o)
    WHERE r.source_doc_id IN $document_ids AND r.claim_id IS NOT NULL
      // 같은 DB에 다른 실습의 그래프가 있어도 이 실습의 개체만 읽습니다.
      AND s.standard_id IN $known_ids AND o.standard_id IN $known_ids
    RETURN r.claim_id AS claim_id, // 문서별 관계를 구분하는 키입니다.
           s.standard_id AS subject_id, s.name AS subject, // 주어의 표준 ID와 이름입니다.
           type(r) AS relation, // 저장한 관계 타입입니다.
           o.standard_id AS object_id, o.name AS object, // 목적어의 표준 ID와 이름입니다.
           r.source_doc_id AS source_doc_id, // 관계의 출처 문서입니다.
           r.evidence AS evidence, r.batch_id AS batch_id // 인용문과 최초 적재 묶음입니다.
    ORDER BY claim_id
    """,
        document_ids=document_ids,
        known_ids=known_ids,
    )

### 2-3. 영화의 기존 상태와 새 장르 노드 준비

DB의 출연 관계를 먼저 조회하고, 새 관계에 사용할 장르 노드를 준비합니다. 기존 관계를 재적재하는 단계가 아닙니다.

In [ ]:
# 조회 범위에는 기존 관계의 출처와 이번 추출의 출처를 모두 넣습니다.
demo_known_ids = [row["standard_id"] for row in demo_expanded_catalog]
demo_document_ids = sorted({row["source_doc_id"] for row in demo_baseline + demo_batch})
demo_before = read_claims(demo_document_ids, demo_known_ids)
print("추가 적재 전 DB 관계:", len(demo_before))
for row in demo_before:
    print("기존 관계:", row["subject"], "->", row["relation"], "->", row["object"])

# 새로 사용할 타입의 표준 ID에도 고유성 제약을 적용합니다.
for entity_type in sorted({row["entity_type"] for row in demo_approved}):
    run_cypher(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{entity_type}) REQUIRE n.standard_id IS UNIQUE"
    )
print("등록을 확인한 새 개체:", put_nodes(demo_approved))

#### 같은 문서의 관계 키 확인하기

같은 영화와 장르라도 다른 문서가 보고하면 다른 기록입니다. 아래 비교는 DB에 쓰지 않습니다.

In [ ]:
key_row = demo_ready[0]
same_row = dict(key_row)
another_document = dict(key_row, source_doc_id="another:document")
print("같은 문서와 관계이면 같은 키:", claim_key(key_row) == claim_key(same_row))
print("다른 문서이면 다른 키:", claim_key(key_row) != claim_key(another_document))

### 🖐️ 함께 따라하기: 의료의 기존 그래프와 새 노드 준비

기존 치료와 완화 관계를 확인하고, 새 증상 노드를 ID로 찾거나 만듭니다.

In [ ]:
# [제공코드]

# 조회 범위에는 기존 관계의 출처와 이번 추출의 출처를 모두 넣습니다.
paper_known_ids = [row["standard_id"] for row in paper_expanded_catalog]
paper_document_ids = sorted({row["source_doc_id"] for row in paper_baseline + paper_batch})
paper_before = read_claims(paper_document_ids, paper_known_ids)
print("추가 적재 전 DB 관계:", len(paper_before))
for row in paper_before:
    print("기존 관계:", row["subject"], "->", row["relation"], "->", row["object"])

# 새로 사용할 타입의 표준 ID에도 고유성 제약을 적용합니다.
for entity_type in sorted({row["entity_type"] for row in paper_approved}):
    run_cypher(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{entity_type}) REQUIRE n.standard_id IS UNIQUE"
    )
print("등록을 확인한 새 개체:", put_nodes(paper_approved))

## 3. 새 관계를 추가하고 재실행합니다

**`put_claims`에 넘기는 것은 이번에 추출하고 검사한 `ready`뿐입니다.** 기존 관계는 그대로 둡니다.  
같은 입력을 다시 저장해도 결과가 달라지지 않는 성질을 **멱등성(idempotency)** 이라고 합니다.

`processed`는 찾거나 처리한 수입니다. 실제로 늘어난 수는 적재 전후 DB 관계 수의 차이로 확인합니다.

### 3-1. 영화에 새 장르 관계 추가

출연 8건에 장르 1건을 추가합니다. 기존 관계의 ID, 출처와 근거가 같은지도 확인합니다.

In [ ]:
# 이번에 새로 만든 관계를 나중에 구분할 적재 묶음 ID입니다.
demo_batch_id = "demo:delta:" + str(uuid4())
demo_processed = put_claims(demo_ready, demo_batch_id, demo_schema_version)
demo_after = read_claims(demo_document_ids, demo_known_ids)
print("처리한 새 추출:", demo_processed)
print("DB 관계 수:", len(demo_before), "->", len(demo_after))
print("실제 추가된 관계:", len(demo_after) - len(demo_before))

# 이전 관계의 ID와 근거까지 같은지 비교합니다.
demo_after_by_id = {row["claim_id"]: row for row in demo_after}
demo_existing_preserved = all(
    demo_after_by_id[row["claim_id"]] == row for row in demo_before
)
print("기존 관계와 근거 유지:", demo_existing_preserved)

#### 적재한 개체의 표준 ID 다시 사용하기

**출처 연결은 저장한 개체에서 원문으로 돌아가는 경로입니다.** `demo_ready`의 확정 ID를 타입과 이름으로 찾을 수 있게 준비합니다. ID를 다시 판정하지 않습니다.

In [ ]:
# ready에는 이미 확정한 양 끝의 표준 ID가 있습니다. ID 후보를 다시 검색하지 않습니다.
demo_linked_ids = {}
for row in demo_ready:
    for role in ["subject", "object"]:
        key = (row[role + "_type"], row[role])
        demo_linked_ids[key] = row[role + "_id"]

demo_stored_entity_ids = set(demo_linked_ids.values())
for (entity_type, name), standard_id in demo_linked_ids.items():
    print("타입:", entity_type, "/ 이름:", name, "/ 표준 ID:", standard_id)

#### 빌더의 출처 연결을 저장할 ID로 옮기기

빌더의 `FROM_CHUNK`에서 개체 ID는 표준 ID로, 청크 ID는 고정 ID로 바꿉니다. 다음 셀에서 이 연결을 DB에 저장합니다.

In [ ]:
# linked_ids는 앞에서 확정한 개체 ID, chunk_id_map은 청크의 임시 ID와 고정 ID를 연결합니다.
demo_source_links = []
for run in demo_packet["builder_runs"]:
    nodes = {node["id"]: node for node in run["graph"]["nodes"]}
    for edge in run["graph"]["relationships"]:
        if edge["type"] != "FROM_CHUNK":
            continue
        node = nodes[edge["start_node_id"]]  # 출처 청크에 연결된 원래 개체입니다.
        key = (node["label"], node["properties"].get("name"))
        if key not in demo_linked_ids:
            continue
        # 기존 출처 연결의 양 끝 ID만 바꿉니다. 새 출처를 추측하지 않습니다.
        demo_source_links.append(
            {
                "entity_type": node["label"],
                "standard_id": demo_linked_ids[key],
                "chunk_id": demo_chunk_id_map[edge["end_node_id"]],
            }
        )
print("출처에 연결할 개체:", len(demo_stored_entity_ids), "/ 연결할 기록:", len(demo_source_links))

#### 개체와 출처 청크 연결하기

`개체 -[:FROM_CHUNK]-> Chunk -[:FROM_DOCUMENT]-> Document`로 원문을 찾아갈 수 있게 합니다.

In [ ]:
demo_source_query = """
UNWIND $rows AS item
MATCH (e:$(item.entity_type) {standard_id: item.standard_id})
MATCH (c:Chunk {chunk_id: item.chunk_id})
MERGE (e)-[:FROM_CHUNK]->(c)
RETURN count(*) AS processed // 기존 연결을 재사용한 기록도 셉니다.
"""
demo_source_result = run_cypher(demo_source_query, rows=demo_source_links)
print("처리한 출처 연결:", demo_source_result[0]["processed"])

#### 저장된 개체의 출처 원문 조회하기

개체에서 청크와 문서를 따라가 원문을 출력합니다. 출처 위치를 보여 주는 연결이며, 관계의 의미는 인용 근거로 판단합니다.

In [ ]:
demo_source_chunk_ids = [row["chunk_id"] for row in demo_chunk_rows]
demo_read_sources_query = """
MATCH (e)-[link:FROM_CHUNK]->(c:Chunk)-[parent:FROM_DOCUMENT]->(d:Document)
WHERE c.chunk_id IN $chunk_ids AND e.standard_id IN $entity_ids
RETURN e.standard_id AS standard_id, // 개체 ID입니다.
       d.doc_id AS doc_id, d.title AS title, d.url AS url, // 출처 문서입니다.
       c.chunk_id AS chunk_id, c.index AS index, c.text AS text, // 출처 청크입니다.
       size(c.embedding) AS dimensions, // 청크의 임베딩 차원입니다.
       elementId(d) AS document_node, elementId(c) AS chunk_node, // 재사용 여부를 비교할 DB ID입니다.
       elementId(link) AS source_link, elementId(parent) AS document_link // 출처 연결도 중복되지 않는지 비교합니다.
ORDER BY standard_id, chunk_id
"""
demo_sources_after = run_cypher(
    demo_read_sources_query,
    chunk_ids=demo_source_chunk_ids,
    entity_ids=list(demo_stored_entity_ids),
)
for source in demo_sources_after:
    print("개체:", source["standard_id"], "/ 문서:", source["doc_id"])
    print("청크 원문:", source["text"])
    print()

#### 문서, 청크와 출처 연결도 다시 저장하기

같은 저장 쿼리를 실행하고 DB 노드, 연결 ID와 원문까지 같은지 비교합니다.

In [ ]:
# 같은 키로 문서, 청크와 출처 연결을 다시 저장합니다.
run_cypher(demo_document_query, rows=demo_document_rows)
run_cypher(demo_chunk_query, rows=demo_chunk_rows)
run_cypher(demo_source_query, rows=demo_source_links)
demo_sources_replayed = run_cypher(
    demo_read_sources_query,
    chunk_ids=demo_source_chunk_ids,
    entity_ids=list(demo_stored_entity_ids),
)
print("문서, 청크와 출처 연결의 ID 및 내용 유지:", demo_sources_after == demo_sources_replayed)

#### 전체 문서와 청크 수 확인하기

개체가 추출되지 않은 청크도 문서의 일부이므로 포함해 셉니다.

In [ ]:
# 개체와 연결되지 않은 청크도 포함해 저장한 전체 문서와 청크를 셉니다.
demo_saved_counts = run_cypher(
    """
MATCH (d:Document)
WHERE d.doc_id IN $doc_ids
OPTIONAL MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d)
WHERE c.chunk_id IN $chunk_ids
RETURN count(DISTINCT d) AS documents, // 문서 노드 수입니다.
       count(DISTINCT c) AS chunks // 이번에 저장한 청크 노드 수입니다.
""",
    doc_ids=[row["doc_id"] for row in demo_document_rows],
    chunk_ids=demo_source_chunk_ids,
)[0]
print("재실행 후 문서:", demo_saved_counts["documents"], "/ 청크:", demo_saved_counts["chunks"])

#### 같은 관계를 다시 저장하기

조회 결과의 건수뿐 아니라 ID와 근거까지 같아야 합니다.

In [ ]:
demo_replay_processed = put_claims(demo_ready, demo_batch_id, demo_schema_version)
demo_replayed = read_claims(demo_document_ids, demo_known_ids)
print("다시 처리한 행:", demo_replay_processed)
print("재실행 전후 관계 수:", len(demo_after), "/", len(demo_replayed))
print("ID와 근거까지 동일:", demo_after == demo_replayed)

### 🖐️ 함께 따라하기: 새 증상 관계 추가와 재실행

`paper_ready`만 적재하세요. 기존 4건이 그대로 남고 새 증상 관계가 추가되어야 합니다.

In [ ]:
# (1) paper_batch_id를 "paper:delta:" + str(uuid4())로 만드세요.
# (2) put_claims(paper_ready, paper_batch_id, paper_schema_version)를 paper_processed로 받으세요.
# (3) read_claims(paper_document_ids, paper_known_ids)로 paper_after를 읽으세요.
# (4) 전후 관계 수와 실제 늘어난 수를 출력하세요.
# (5) claim_id로 비교해 paper_before의 ID와 근거가 paper_after에 그대로 있는지 확인하세요.
# 여기에 코드를 작성하세요.

#### 적재한 의료 개체의 표준 ID 준비

`paper_ready`의 확정 ID를 출처 연결에 재사용합니다.

In [ ]:
# [제공코드]

# ready에는 이미 확정한 양 끝의 표준 ID가 있습니다. ID 후보를 다시 검색하지 않습니다.
paper_linked_ids = {}
for row in paper_ready:
    for role in ["subject", "object"]:
        key = (row[role + "_type"], row[role])
        paper_linked_ids[key] = row[role + "_id"]

paper_stored_entity_ids = set(paper_linked_ids.values())
for (entity_type, name), standard_id in paper_linked_ids.items():
    print("타입:", entity_type, "/ 이름:", name, "/ 표준 ID:", standard_id)

#### 의료 개체의 원래 출처 연결 준비

빌더가 기록한 출처 연결의 양 끝을 표준 개체 ID와 고정 청크 ID로 바꿉니다.

In [ ]:
# [제공코드]

# linked_ids는 앞에서 확정한 개체 ID, chunk_id_map은 청크의 임시 ID와 고정 ID를 연결합니다.
paper_source_links = []
for run in paper_packet["builder_runs"]:
    nodes = {node["id"]: node for node in run["graph"]["nodes"]}
    for edge in run["graph"]["relationships"]:
        if edge["type"] != "FROM_CHUNK":
            continue
        node = nodes[edge["start_node_id"]]  # 출처 청크에 연결된 원래 개체입니다.
        key = (node["label"], node["properties"].get("name"))
        if key not in paper_linked_ids:
            continue
        # 기존 출처 연결의 양 끝 ID만 바꿉니다. 새 출처를 추측하지 않습니다.
        paper_source_links.append(
            {
                "entity_type": node["label"],
                "standard_id": paper_linked_ids[key],
                "chunk_id": paper_chunk_id_map[edge["end_node_id"]],
            }
        )
print("출처에 연결할 개체:", len(paper_stored_entity_ids), "/ 연결할 기록:", len(paper_source_links))

#### 의료 개체와 청크 연결하기

`paper_source_query`를 만들고 `paper_source_links`를 전달하세요. 노드 라벨은 각 행의 `entity_type`입니다.

In [ ]:
# (1) UNWIND $rows로 행을 꺼내고, 타입과 standard_id로 개체를 찾으세요.
# (2) chunk_id로 청크를 찾고 (e)-[:FROM_CHUNK]->(c)를 MERGE하세요.
# (3) count(*) AS processed를 반환하고 paper_source_result로 결과를 받아 출력하세요.
# 여기에 코드를 작성하세요.

#### 의료 개체에서 원문으로 돌아가기

문서 ID와 청크 원문을 출력하고, 재실행 비교에 사용할 DB 식별자도 함께 읽습니다.

In [ ]:
# [제공코드]

paper_source_chunk_ids = [row["chunk_id"] for row in paper_chunk_rows]
paper_read_sources_query = """
MATCH (e)-[link:FROM_CHUNK]->(c:Chunk)-[parent:FROM_DOCUMENT]->(d:Document)
WHERE c.chunk_id IN $chunk_ids AND e.standard_id IN $entity_ids
RETURN e.standard_id AS standard_id, // 개체 ID입니다.
       d.doc_id AS doc_id, d.title AS title, d.url AS url, // 출처 문서입니다.
       c.chunk_id AS chunk_id, c.index AS index, c.text AS text, // 출처 청크입니다.
       size(c.embedding) AS dimensions, // 청크의 임베딩 차원입니다.
       elementId(d) AS document_node, elementId(c) AS chunk_node, // 재사용 여부를 비교할 DB ID입니다.
       elementId(link) AS source_link, elementId(parent) AS document_link // 출처 연결도 중복되지 않는지 비교합니다.
ORDER BY standard_id, chunk_id
"""
paper_sources_after = run_cypher(
    paper_read_sources_query,
    chunk_ids=paper_source_chunk_ids,
    entity_ids=list(paper_stored_entity_ids),
)
for source in paper_sources_after:
    print("개체:", source["standard_id"], "/ 문서:", source["doc_id"])
    print("청크 원문:", source["text"])
    print()

#### 의료 문서와 청크의 재사용 확인

문서 1개와 청크 1개가 계속 유지되고, 출처 연결의 ID와 원문도 그대로인지 확인합니다.

In [ ]:
# (1) document_query, chunk_query, source_query에 앞의 같은 입력을 넣어 다시 실행하세요.
# (2) paper_read_sources_query를 같은 chunk_ids와 entity_ids로 실행해 paper_sources_replayed에 받으세요.
# (3) paper_sources_after == paper_sources_replayed를 출력해 ID와 원문이 유지됐는지 확인하세요.
# 여기에 코드를 작성하세요.

#### 의료 문서와 전체 청크 수 확인하기

개체와 연결되지 않은 청크까지 포함해 셉니다.

In [ ]:
# [제공코드]

# 개체와 연결되지 않은 청크도 포함해 저장한 전체 문서와 청크를 셉니다.
paper_saved_counts = run_cypher(
    """
MATCH (d:Document)
WHERE d.doc_id IN $doc_ids
OPTIONAL MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d)
WHERE c.chunk_id IN $chunk_ids
RETURN count(DISTINCT d) AS documents, // 문서 노드 수입니다.
       count(DISTINCT c) AS chunks // 이번에 저장한 청크 노드 수입니다.
""",
    doc_ids=[row["doc_id"] for row in paper_document_rows],
    chunk_ids=paper_source_chunk_ids,
)[0]
print("재실행 후 문서:", paper_saved_counts["documents"], "/ 청크:", paper_saved_counts["chunks"])

#### 의료 관계도 같은 입력으로 재실행하기

본문의 영화와 같은 방식으로 재실행 결과를 비교하세요.

In [ ]:
# (1) 같은 paper_ready와 batch_id, schema_version으로 put_claims를 다시 호출하세요.
# (2) 반환값은 paper_replay_processed, DB 조회 결과는 paper_replayed에 담으세요.
# (3) paper_after == paper_replayed와 전후 관계 수를 출력하세요.
# 여기에 코드를 작성하세요.

## 4. 서로 다른 개념을 계층으로 연결해 조회합니다

**좁은 개념과 넓은 개념은 같은 개체가 아닙니다.** `supernatural horror`를 `horror`와 합치면 장르의 세부 구분을 잃습니다.

- **계층 관계:** `하위 개념 -[:BROADER]-> 상위 개념`
- **조회:** 상위 개념으로 질문할 때 하위 개념에 연결된 결과도 탐색합니다.
- **출처:** 계층은 검토해 정한 모델링 규칙이며, LLM이 새로 추출한 사실과 구분합니다.

`BROADER`는 이 자료에서 정한 관계 이름입니다. 계층을 추가해도 원래 관계의 목적어와 근거는 바뀌지 않습니다.

#### 상하 개념을 연결하는 함수

상위 개념 노드를 먼저 준비하고 `BROADER`를 연결합니다. 처리 수는 이미 존재한 연결도 포함합니다.

In [ ]:
def put_hierarchy(edges):
    """하위 개념에서 상위 개념으로 향하는 BROADER 관계를 만듭니다.

    Args:
        edges: child_id, parent_id와 상위 개념의 이름, 타입, 정의 이유를 담은 목록.
    Returns:
        처리한 계층 관계 수. 이미 있던 관계도 포함합니다.
    """
    # 상위 개념 노드가 없으면 먼저 만듭니다.
    run_cypher(
        """
    UNWIND $edges AS item
    MERGE (parent:$(item.parent_type) {standard_id: item.parent_id})
    ON CREATE SET parent.name = item.parent_name
    RETURN count(parent) AS processed // 이미 있던 개념도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    # 상위 개념을 모두 준비한 뒤 하위 개념과 연결합니다.
    rows = run_cypher(
        """
    UNWIND $edges AS item
    // ID로 하위 개념과 상위 개념 노드를 찾습니다.
    MATCH (child {standard_id: item.child_id})
    MATCH (parent {standard_id: item.parent_id})
    // 원문에서 추출한 관계와 구분해, 개념의 상하 관계는 BROADER로 연결합니다.
    MERGE (child)-[r:BROADER]->(parent)
    ON CREATE SET r.note = item.note // 왜 이 계층을 정했는지 남깁니다.
    RETURN count(r) AS processed // 이미 있던 관계도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    return rows[0]["processed"]

### 4-1. 상위 장르로 영화 찾기

`*0..`는 같은 개념 또는 `BROADER`로 이어진 상위 개념까지 탐색합니다. 반복 실행으로 계층이 이미 있다면 연결 전부터 같은 결과가 나옵니다.

In [ ]:
demo_top_id = "movies:genre:horror"
demo_broad_query = """
MATCH (m:Movie)-[r:HAS_GENRE]->(g:Genre)-[:BROADER*0..]->(:Genre {standard_id: $top_id})
WHERE r.claim_id IS NOT NULL AND m.standard_id IN $known_ids
RETURN DISTINCT m.name AS movie // 하위 장르를 포함해 찾은 영화입니다.
ORDER BY movie
"""
demo_before_rows = run_cypher(
    demo_broad_query, top_id=demo_top_id, known_ids=demo_known_ids
)
print("계층 연결 전:", demo_before_rows)

#### 장르 계층 추가 후 같은 질문하기

장르를 합치지 않고도 `horror`로 `The Devil's Advocate`를 찾습니다.

In [ ]:
# demo_hierarchy.json: 원문 추출과 별도로 정한 개념의 상하 관계입니다.
demo_hierarchy = read_json("demo_hierarchy.json")
demo_hierarchy_processed = put_hierarchy(demo_hierarchy)
demo_after_rows = run_cypher(demo_broad_query, top_id=demo_top_id, known_ids=demo_known_ids)
print("처리한 계층 관계:", demo_hierarchy_processed)
print("계층 연결 후:", demo_after_rows)

### 🖐️ 함께 따라하기: 기존 치료 관계를 질환 계층으로 조회합니다

기존 `plaque psoriasis` 치료 관계를 유지하면서 `psoriasis`로도 찾습니다. 새로운 증상 관계와 별개로, 기존 그래프도 계속 조회할 수 있습니다.

In [ ]:
# [제공코드]

paper_top_id = "Disease::DOID:8893"
paper_broad_query = """
MATCH (c:Compound)-[r:TREATS]->(d:Disease)-[:BROADER*0..]->(:Disease {standard_id: $top_id})
WHERE r.claim_id IS NOT NULL AND c.standard_id IN $known_ids
RETURN DISTINCT c.name AS compound, d.name AS disease // 원래 치료 관계의 약물과 질환입니다.
ORDER BY compound, disease
"""
paper_before_rows = run_cypher(
    paper_broad_query, top_id=paper_top_id, known_ids=paper_known_ids
)
print("계층 연결 전:", paper_before_rows)

#### 질환 계층 연결 후 조회하기

`paper_hierarchy.json`을 적용하고 같은 질의를 다시 실행하세요. 원래 치료 대상은 `plaque psoriasis`로 남아 있습니다.

In [ ]:
# (1) paper_hierarchy.json을 paper_hierarchy에 읽으세요.
# (2) put_hierarchy(paper_hierarchy)의 반환값을 paper_hierarchy_processed에 담으세요.
# (3) paper_broad_query를 같은 파라미터로 실행해 paper_after_rows에 담으세요.
# (4) 처리 수와 계층 연결 후 결과를 출력하세요.
# 여기에 코드를 작성하세요.

## 5. 처리 이력을 남기고 이번 추가만 되돌립니다

**추출 완료와 적재 완료는 다릅니다.** `ProcessingState`에 문서별 최근 적재 버전과 건수를 기록합니다.  
`stored`는 적재 가능한 행의 처리가 끝났다는 뜻이며, 보류나 기각의 해결까지 뜻하지 않습니다.

**되돌리기는 이번 `batch_id`로 새로 만든 관계만 삭제합니다.** 기존 추출 관계, 문서와 청크를 포함한 모든 노드, 출처 연결과 계층은 유지합니다.  
여러 함수의 쿼리는 각각 실행되므로, 이 예제는 여러 단계가 한꺼번에 취소되는 트랜잭션은 아닙니다.

#### 문서별 적재 이력 기록 함수

최근 규칙 버전, 적재 가능 수, 보류 수, 기각 수와 묶음 ID를 기록합니다.

In [ ]:
def record_state(document, version, ready, pending, rejected, batch_id):
    """문서별 마지막 적재 처리의 버전, 검사 건수와 상태를 기록합니다.

    Args:
        document: 출처 doc_id가 있는 원문.
        version: 검사에 적용한 규칙 버전.
        ready, pending, rejected: 해당 문서의 검사 결과 목록.
        batch_id: 이번 적재 묶음 ID.
    Returns:
        문서 ID, 버전, 적재 상태와 검사 건수의 딕셔너리.
    """
    rows = run_cypher(
        """
    MERGE (d:ProcessingState {doc_id: $doc_id})
    SET d.schema_version = $version, d.status = 'stored', d.batch_id = $batch_id,
        d.accepted = $accepted, d.pending = $pending, d.rejected = $rejected,
        d.processed_at = datetime()
    RETURN d.doc_id AS doc_id, d.schema_version AS schema_version, // 문서와 적용 규칙입니다.
           d.status AS status, // stored는 적재 가능 행의 저장을 마친 상태입니다.
           d.accepted AS accepted, d.pending AS pending, d.rejected AS rejected // 이번 검사 건수입니다.
    """,
        doc_id=document["doc_id"],
        version=version,
        batch_id=batch_id,
        accepted=len(ready),
        pending=len(pending),
        rejected=len(rejected),
    )
    return rows[0]

### 5-1. 영화의 새 관계 적재 이력

이번 추출을 실행한 문서만 기록합니다. 결과가 0건인 문서도 처리 완료 여부를 남깁니다.

In [ ]:
run_cypher(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (d:ProcessingState) REQUIRE d.doc_id IS UNIQUE"
)
demo_storage_states = []
# 관계가 0건이어도 추출을 완료한 문서는 처리 이력에 남깁니다.
for run in demo_packet["builder_runs"]:
    document = run["document"]
    doc_id = document["doc_id"]
    ready_rows = [row for row in demo_ready if row["source_doc_id"] == doc_id]
    pending_rows = [row for row in demo_pending if row["source_doc_id"] == doc_id]
    rejected_rows = [row for row in demo_rejected if row["source_doc_id"] == doc_id]
    state = record_state(
        document,
        demo_schema_version,
        ready_rows,
        pending_rows,
        rejected_rows,
        demo_batch_id,
    )
    demo_storage_states.append(state)
    pprint(state)

#### 영화의 이번 추가만 되돌리기

새 장르 관계를 되돌리고 기존 출연 관계가 그대로인지 비교합니다.

In [ ]:
# 이번 batch_id로 처음 만든 관계만 지웁니다. 이전 관계, 노드와 BROADER는 유지합니다.
demo_removed = run_cypher(
    """
MATCH ()-[r]->()
WHERE r.batch_id = $batch_id
DELETE r
RETURN count(*) AS removed // 이번 묶음에서 삭제한 관계 수입니다.
""",
    batch_id=demo_batch_id,
)[0]
demo_restored = read_claims(demo_document_ids, demo_known_ids)
print("되돌린 관계 수:", demo_removed["removed"])
print("남은 기존 관계:", len(demo_restored))
print("기존 관계와 근거 복원:", demo_restored == demo_before)

# 관계가 0건인 문서의 완료 상태도 이번 묶음과 함께 되돌립니다.
demo_reverted_states = run_cypher(
    """
MATCH (d:ProcessingState {batch_id: $batch_id})
SET d.status = 'reverted'
RETURN d.doc_id AS doc_id, d.status AS status // 추가 적재를 되돌린 문서입니다.
ORDER BY doc_id
""",
    batch_id=demo_batch_id,
)
pprint(demo_reverted_states)


# 추출 관계를 되돌려도 문서, 청크와 개체의 출처 연결은 남깁니다.
demo_sources_restored = run_cypher(
    demo_read_sources_query,
    chunk_ids=demo_source_chunk_ids,
    entity_ids=list(demo_stored_entity_ids),
)
print("되돌린 후에도 출처 유지:", demo_sources_after == demo_sources_restored)

### 🖐️ 함께 따라하기: 의료 적재 이력과 되돌리기

새 증상 관계의 적재 이력을 남기고 이번 묶음만 제거하세요. 기존 치료와 완화 4건이 남아야 합니다.

In [ ]:
# (1) ProcessingState.doc_id의 고유성 제약을 만들고 paper_storage_states를 준비하세요.
# (2) paper_packet["builder_runs"]의 각 document를 꺼내세요.
# (3) 해당 문서의 ready, pending, rejected를 골라 record_state에 넘기세요.
# (4) 반환값을 paper_storage_states에 담고 출력하세요.
# 여기에 코드를 작성하세요.

#### 의료의 이번 추가만 되돌리기

이번 묶음으로 만든 관계만 제거하고 처리 상태를 `reverted`로 바꿉니다.

In [ ]:
# [제공코드]

# 이번 batch_id로 처음 만든 관계만 지웁니다. 이전 관계, 노드와 BROADER는 유지합니다.
paper_removed = run_cypher(
    """
MATCH ()-[r]->()
WHERE r.batch_id = $batch_id
DELETE r
RETURN count(*) AS removed // 이번 묶음에서 삭제한 관계 수입니다.
""",
    batch_id=paper_batch_id,
)[0]
paper_restored = read_claims(paper_document_ids, paper_known_ids)
print("되돌린 관계 수:", paper_removed["removed"])
print("남은 기존 관계:", len(paper_restored))
print("기존 관계와 근거 복원:", paper_restored == paper_before)

# 관계가 0건인 문서의 완료 상태도 이번 묶음과 함께 되돌립니다.
paper_reverted_states = run_cypher(
    """
MATCH (d:ProcessingState {batch_id: $batch_id})
SET d.status = 'reverted'
RETURN d.doc_id AS doc_id, d.status AS status // 추가 적재를 되돌린 문서입니다.
ORDER BY doc_id
""",
    batch_id=paper_batch_id,
)
pprint(paper_reverted_states)


# 추출 관계를 되돌려도 문서, 청크와 개체의 출처 연결은 남깁니다.
paper_sources_restored = run_cypher(
    paper_read_sources_query,
    chunk_ids=paper_source_chunk_ids,
    entity_ids=list(paper_stored_entity_ids),
)
print("되돌린 후에도 출처 유지:", paper_sources_after == paper_sources_restored)

#### 연결 종료

새 관계 파일과 기존 DB 관계는 유지합니다.

In [ ]:
driver.close()

## 교안 02 핵심 코드 이어서 보기

의료 자료로 **ID 연결 -> 문서와 청크 저장 -> 새 관계와 출처 연결 -> 재실행 -> 계층 조회 -> 이력과 되돌리기**를 이어 실행합니다. 교안 01이 준비한 기존 DB 관계와 `paper_extraction_packet.json`이 필요합니다.

### 1. 새 관계의 이름을 표준 ID에 연결합니다

#### 1-1. 연결과 새 관계 파일 읽기

#### 라이브러리와 Neo4j 연결

`.env`의 접속 정보로 연결하고, `run_cypher`로 쿼리를 실행합니다. 데이터와 결과 파일의 경로도 준비합니다.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
from pprint import pprint
from uuid import uuid4
from urllib.parse import urlsplit

from dotenv import find_dotenv, load_dotenv
from neo4j import GraphDatabase

# 학생용은 현재 폴더, 정답은 한 단계 위 폴더의 자료를 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)


def read_json(name):
    """data 폴더의 JSON 파일을 목록 또는 딕셔너리로 읽습니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """처리 결과를 output 폴더에 한글을 유지해 저장합니다."""
    (output_dir / name).write_text(
        json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8"
    )


# 현재 작업 폴더부터 상위로 올라가 가장 가까운 .env를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
neo4j_uri = os.environ["NEO4J_URI"]
# driver는 여러 쿼리에서 재사용할 DB 연결 통로입니다. 계정 정보는 출력하지 않습니다.
driver = GraphDatabase.driver(
    neo4j_uri,
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
# 연결 객체 생성만으로 접속 성공이 보장되지 않으므로 지금 서버 접속을 확인합니다.
driver.verify_connectivity()


def run_cypher(query, **params):
    """값을 매개변수로 전달하고 Cypher 결과를 딕셔너리 리스트로 돌려줍니다."""
    # 쿼리마다 세션을 열고 with 블록이 끝나면 닫습니다. driver는 계속 재사용합니다.
    with driver.session() as session:
        # RETURN에서 붙인 별칭이 딕셔너리 키가 되어 파이썬에서 조회할 수 있습니다.
        return [record.data() for record in session.run(query, **params)]


# 주소에 계정 정보가 포함되어 있어도 호스트와 포트만 확인합니다.
connection_address = urlsplit(neo4j_uri)
print("Neo4j 연결 완료. 호스트:", connection_address.hostname, "/ 포트:", connection_address.port)

#### 입력 파일 읽기

In [ ]:
# paper_extraction_packet.json: 앞 교안에서 만든 새 관계, 원문과 검사 결과입니다.
paper_packet = json.loads(
    (output_dir / "paper_extraction_packet.json").read_text(encoding="utf-8")
)
paper_batch = paper_packet["rows"]  # 검사 전 새 추출 전체입니다.
paper_validated = paper_packet["validated"]  # 두 검사를 통과해 ID를 연결할 행입니다.
paper_rejected = paper_packet["rejected"]  # 교안 01의 기각 사유를 그대로 보존합니다.
paper_docs = paper_packet["documents"]
paper_signatures = paper_packet["signatures"]
paper_schema_version = paper_packet["schema_version"]

# 기존 그래프의 개체 목록입니다. 이 셀에서 기존 관계를 다시 저장하지 않습니다.
paper_existing = read_json(paper_packet["baseline_file"])
paper_catalog = paper_existing["catalog"]
paper_baseline = paper_existing["rows"]
print("파일에서 받은 새 관계:", len(paper_batch))
print("검사 통과:", len(paper_validated), "/ 검사 기각:", len(paper_rejected))
for row in paper_validated:
    print("ID를 연결할 관계:", (row["subject"], row["relation"], row["object"]))

#### 1-2. 검토한 새 증상 목록 추가

In [ ]:
# paper_new_entities.json: 원문으로 확인한 새 개체의 이름, 별칭과 내부 표준 ID입니다.
paper_approved = read_json("paper_new_entities.json")
paper_expanded_catalog = paper_catalog + paper_approved
for entity in paper_approved:
    print("이름:", entity["name"], "/ 표준 ID:", entity["standard_id"])
    print("확인한 내용:", entity["review_note"])
print("기존 개체:", len(paper_catalog), "/ 보완한 목록:", len(paper_expanded_catalog))

#### 1-3. 타입과 이름별 ID 조회 준비

In [ ]:
# (타입, 별칭)을 키로 사용합니다. 이름이 같아도 타입이 다르면 다른 키입니다.
paper_ids_by_name = {}
for entity in paper_expanded_catalog:
    for alias in entity["aliases"]:
        key = (entity["entity_type"], alias)
        # 같은 ID가 중복 등록되어도 후보 하나로 세도록 집합에 담습니다.
        paper_ids_by_name.setdefault(key, set()).add(entity["standard_id"])

# 첫 관계의 두 이름으로 조회 결과를 확인합니다.
for role in ("subject", "object"):
    row = paper_validated[0]
    key = (row[role + "_type"], row[role])
    print("타입과 이름:", key, "/ 후보 ID:", sorted(paper_ids_by_name.get(key, set())))

#### 1-4. 양 끝 ID 연결과 보류 분류

In [ ]:
paper_ready, paper_pending = [], []
# 교안 01의 검사 통과 행에만 ID를 연결합니다. 원본 대신 복사본에 기록합니다.
for original in paper_validated:
    row = dict(original)
    unresolved = []
    for role in ("subject", "object"):
        key = (row[role + "_type"], row[role])
        candidates = paper_ids_by_name.get(key, set())
        if len(candidates) == 1:
            row[role + "_id"] = next(iter(candidates))  # 하나뿐인 ID를 꺼냅니다.
        else:
            unresolved.append(row[role])  # 후보가 없거나 여러 개이면 보류합니다.

    if unresolved:
        paper_pending.append(dict(row, unresolved_names=unresolved))
    else:
        paper_ready.append(row)

print("적재 가능:", len(paper_ready), "/ ID 보류:", len(paper_pending), "/ 검사 기각:", len(paper_rejected))
for row in paper_ready:
    print("표준 ID 연결:", row["subject_id"], "->", row["relation"], "->", row["object_id"])
for row in paper_pending:
    print("ID를 확인할 이름:", row["unresolved_names"])

### 2. 문서와 청크를 재사용하고 적재할 노드를 준비합니다

#### 2-1. 문서 정보와 청크의 고정 키 준비

In [ ]:
# builder_runs에는 교안 01에서 추출한 문서와 그래프가 함께 들어 있습니다.
paper_document_rows = []
for run in paper_packet["builder_runs"]:
    document = run["document"]
    paper_document_rows.append(
        {
            "doc_id": document["doc_id"],
            "title": document["title"],
            "url": document.get("url", ""),
        }
    )
for document in paper_document_rows:
    print("문서 ID:", document["doc_id"], "/ 제목:", document["title"])

#### 같은 입력에서 같은 청크 ID 만들기

In [ ]:
paper_chunk_rows = []
paper_chunk_id_map = {}  # 빌더의 임시 청크 ID를 DB에서 재사용할 고정 ID에 연결합니다.
for run in paper_packet["builder_runs"]:
    doc_id = run["document"]["doc_id"]
    for node in run["graph"]["nodes"]:
        if node["label"] != "Chunk":
            continue
        chunk = node["properties"]
        # 문서와 순번이 같아도 청크 원문이 바뀌면 다른 ID가 됩니다.
        text_hash = hashlib.sha256(chunk["text"].encode("utf-8")).hexdigest()
        chunk_id = f"{doc_id}:chunk:{chunk['index']}:{text_hash}"
        paper_chunk_id_map[node["id"]] = chunk_id
        paper_chunk_rows.append(
            {
                "chunk_id": chunk_id,
                "doc_id": doc_id,
                "index": chunk["index"],
                "text": chunk["text"],
                "embedding": node["embedding_properties"]["embedding"],
                "embedding_model": run["settings"]["embedding_model"],
            }
        )

#### 청크 원문과 임베딩 확인

In [ ]:
for chunk in paper_chunk_rows:
    print("문서:", chunk["doc_id"], "/ 청크 순번:", chunk["index"])
    print("원문:", chunk["text"])
    print("임베딩 차원:", len(chunk["embedding"]))
    print()

#### 2-2. 문서와 청크를 중복 없이 저장

In [ ]:
# 같은 문서 ID를 가진 Document가 두 개 생기지 않도록 제약을 만듭니다.
run_cypher("CREATE CONSTRAINT IF NOT EXISTS FOR (d:Document) REQUIRE d.doc_id IS UNIQUE")
paper_document_query = """
UNWIND $rows AS item
// 기존 문서가 있으면 찾고, 없을 때만 만듭니다.
MERGE (d:Document {doc_id: item.doc_id})
ON CREATE SET d.title = item.title, d.url = item.url
RETURN d.doc_id AS doc_id, d.title AS title // 찾거나 만든 문서입니다.
ORDER BY doc_id
"""
paper_stored_documents = run_cypher(paper_document_query, rows=paper_document_rows)
print("저장하거나 재사용한 문서:", paper_stored_documents)

#### 기존 임베딩을 가진 청크 저장

In [ ]:
run_cypher("CREATE CONSTRAINT IF NOT EXISTS FOR (c:Chunk) REQUIRE c.chunk_id IS UNIQUE")
paper_chunk_query = """
UNWIND $rows AS item
MATCH (d:Document {doc_id: item.doc_id})
MERGE (c:Chunk {chunk_id: item.chunk_id})
// 처음 저장할 때만 원문과 기존 임베딩을 기록합니다. 임베딩 API를 다시 호출하지 않습니다.
ON CREATE SET c.text = item.text, c.index = item.index,
              c.embedding = item.embedding, c.embedding_model = item.embedding_model
MERGE (c)-[:FROM_DOCUMENT]->(d)
RETURN c.chunk_id AS chunk_id, c.index AS index, // 청크의 고정 ID와 순번입니다.
       size(c.embedding) AS dimensions // 저장한 벡터의 길이입니다.
ORDER BY chunk_id
"""
paper_stored_chunks = run_cypher(paper_chunk_query, rows=paper_chunk_rows)
print("저장하거나 재사용한 청크:", len(paper_stored_chunks))
for chunk in paper_stored_chunks:
    print("순번:", chunk["index"], "/ 임베딩 차원:", chunk["dimensions"])

#### 2-3. 노드, 관계 키와 저장 함수 준비

#### 새 개체의 표준 노드 준비

`entity_type`을 레이블로, `standard_id`를 식별자로 사용합니다. 같은 ID가 있으면 기존 노드를 사용합니다.

In [ ]:
def put_nodes(catalog):
    """개체 목록의 타입과 표준 ID로 노드를 찾거나 만듭니다.

    Args:
        catalog: standard_id, entity_type, name, aliases를 담은 개체 목록.
    Returns:
        처리한 개체 수. 이미 있던 노드도 포함합니다.
    """
    rows = run_cypher(
        """
    UNWIND $catalog AS item
    // 원래 타입을 레이블로 사용하고 표준 ID로 같은 개체를 찾습니다.
    MERGE (n:$(item.entity_type) {standard_id: item.standard_id})
    // 기존 이름과 별칭은 덮어쓰지 않습니다.
    ON CREATE SET n.name = item.name, n.aliases = item.aliases
    RETURN count(n) AS processed // 새로 만든 수가 아닌 처리한 개체 수입니다.
    """,
        catalog=catalog,
    )
    return rows[0]["processed"]

#### 출처를 포함한 관계 키 만들기

문서 ID, 주어 ID, 관계, 목적어 ID를 같은 순서로 해시해 `claim_id`를 만듭니다. 근거 문구를 바꿔도 이 네 값이 같으면 같은 키입니다.

In [ ]:
def claim_key(row):
    """문서 ID와 표준 ID로 표현한 관계를 저장용 키로 바꿉니다.

    Args:
        row: source_doc_id, subject_id, relation, object_id가 있는 기록.
    Returns:
        같은 문서의 같은 관계이면 동일한 문자열 ID.
    """
    values = [
        row["source_doc_id"],
        row["subject_id"],
        row["relation"],
        row["object_id"],
    ]
    # 네 값을 같은 순서로 문자열로 만든 뒤, 같은 입력에 같은 해시값을 얻습니다.
    text = json.dumps(values, ensure_ascii=False)
    return "claim:" + hashlib.sha256(text.encode("utf-8")).hexdigest()

#### 새 관계를 중복 없이 저장하기

`MERGE`로 같은 관계 키를 찾습니다. `ON CREATE SET`은 처음 만든 관계에만 근거와 묶음 ID를 써서 기존 기록을 유지합니다. 반환값은 신규 생성 수가 아닌 처리 수입니다.

In [ ]:
def put_claims(rows, batch_id, schema_version):
    """기존 노드 사이에 문서별 관계와 근거를 중복 없이 저장합니다.

    Args:
        rows: 양 끝 표준 ID를 붙인 추출 기록.
        batch_id: 이번 적재 묶음을 구분하는 ID.
        schema_version: 검사에 사용한 스키마 버전 번호.
    Returns:
        양 끝 노드를 찾아 처리한 기록 수. 기존 관계를 찾은 경우도 셉니다.
    """
    records = []
    for row in rows:
        records.append(dict(row, claim_id=claim_key(row)))
    result = run_cypher(
        """
    UNWIND $rows AS item
    // 먼저 저장해 둔 노드를 타입과 표준 ID로 찾습니다.
    MATCH (s:$(item.subject_type) {standard_id: item.subject_id})
    MATCH (o:$(item.object_type) {standard_id: item.object_id})
    // 같은 문서의 같은 관계이면 기존 관계를 재사용합니다.
    MERGE (s)-[r:$(item.relation) {claim_id: item.claim_id}]->(o)
    // 새 관계에만 출처와 근거를 기록해 이전 문서의 근거를 보존합니다.
    ON CREATE SET r.source_doc_id = item.source_doc_id,
                  r.evidence = item.evidence, r.batch_id = $batch_id,
                  r.schema_version = $schema_version
    RETURN count(r) AS processed // 이미 저장되어 있던 관계도 처리 수에 포함합니다.
    """,
        rows=records,
        batch_id=batch_id,
        schema_version=schema_version,
    )
    return result[0]["processed"]

#### 기존 관계와 새 관계 함께 조회하기

정해 둔 문서와 표준 ID 범위의 관계를 `claim_id` 순서로 읽습니다. 출처별 기록과 근거를 대조할 때 사용합니다.

In [ ]:
def read_claims(document_ids, known_ids):
    """지정한 문서들에서 내가 등록한 개체 사이의 관계와 근거를 읽습니다.

    Args:
        document_ids: 조회할 출처 문서 ID 목록.
        known_ids: 조회에 포함할 표준 ID 목록.
    Returns:
        관계와 출처를 담은 딕셔너리의 리스트. claim_id 순서로 정렬합니다.
    """
    return run_cypher(
        """
    MATCH (s)-[r]->(o)
    WHERE r.source_doc_id IN $document_ids AND r.claim_id IS NOT NULL
      // 같은 DB에 다른 실습의 그래프가 있어도 이 실습의 개체만 읽습니다.
      AND s.standard_id IN $known_ids AND o.standard_id IN $known_ids
    RETURN r.claim_id AS claim_id, // 문서별 관계를 구분하는 키입니다.
           s.standard_id AS subject_id, s.name AS subject, // 주어의 표준 ID와 이름입니다.
           type(r) AS relation, // 저장한 관계 타입입니다.
           o.standard_id AS object_id, o.name AS object, // 목적어의 표준 ID와 이름입니다.
           r.source_doc_id AS source_doc_id, // 관계의 출처 문서입니다.
           r.evidence AS evidence, r.batch_id AS batch_id // 인용문과 최초 적재 묶음입니다.
    ORDER BY claim_id
    """,
        document_ids=document_ids,
        known_ids=known_ids,
    )

#### 2-4. 기존 상태 확인과 새 노드 준비

In [ ]:
# 조회 범위에는 기존 관계의 출처와 이번 추출의 출처를 모두 넣습니다.
paper_known_ids = [row["standard_id"] for row in paper_expanded_catalog]
paper_document_ids = sorted({row["source_doc_id"] for row in paper_baseline + paper_batch})
paper_before = read_claims(paper_document_ids, paper_known_ids)
print("추가 적재 전 DB 관계:", len(paper_before))
for row in paper_before:
    print("기존 관계:", row["subject"], "->", row["relation"], "->", row["object"])

# 새로 사용할 타입의 표준 ID에도 고유성 제약을 적용합니다.
for entity_type in sorted({row["entity_type"] for row in paper_approved}):
    run_cypher(
        f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{entity_type}) REQUIRE n.standard_id IS UNIQUE"
    )
print("등록을 확인한 새 개체:", put_nodes(paper_approved))

### 3. 새 관계를 추가하고 재실행합니다

#### 3-1. 새 관계 추가와 기존 기록 확인

In [ ]:
# 이번에 새로 만든 관계를 나중에 구분할 적재 묶음 ID입니다.
paper_batch_id = "paper:delta:" + str(uuid4())
paper_processed = put_claims(paper_ready, paper_batch_id, paper_schema_version)
paper_after = read_claims(paper_document_ids, paper_known_ids)
print("처리한 새 추출:", paper_processed)
print("DB 관계 수:", len(paper_before), "->", len(paper_after))
print("실제 추가된 관계:", len(paper_after) - len(paper_before))

# 이전 관계의 ID와 근거까지 같은지 비교합니다.
paper_after_by_id = {row["claim_id"]: row for row in paper_after}
paper_existing_preserved = all(
    paper_after_by_id[row["claim_id"]] == row for row in paper_before
)
print("기존 관계와 근거 유지:", paper_existing_preserved)

#### 3-2. 개체에서 출처 청크로 연결

In [ ]:
# ready에는 이미 확정한 양 끝의 표준 ID가 있습니다. ID 후보를 다시 검색하지 않습니다.
paper_linked_ids = {}
for row in paper_ready:
    for role in ["subject", "object"]:
        key = (row[role + "_type"], row[role])
        paper_linked_ids[key] = row[role + "_id"]

paper_stored_entity_ids = set(paper_linked_ids.values())
for (entity_type, name), standard_id in paper_linked_ids.items():
    print("타입:", entity_type, "/ 이름:", name, "/ 표준 ID:", standard_id)

#### 기존 출처 연결의 양 끝 ID 변환

In [ ]:
# linked_ids는 앞에서 확정한 개체 ID, chunk_id_map은 청크의 임시 ID와 고정 ID를 연결합니다.
paper_source_links = []
for run in paper_packet["builder_runs"]:
    nodes = {node["id"]: node for node in run["graph"]["nodes"]}
    for edge in run["graph"]["relationships"]:
        if edge["type"] != "FROM_CHUNK":
            continue
        node = nodes[edge["start_node_id"]]  # 출처 청크에 연결된 원래 개체입니다.
        key = (node["label"], node["properties"].get("name"))
        if key not in paper_linked_ids:
            continue
        # 기존 출처 연결의 양 끝 ID만 바꿉니다. 새 출처를 추측하지 않습니다.
        paper_source_links.append(
            {
                "entity_type": node["label"],
                "standard_id": paper_linked_ids[key],
                "chunk_id": paper_chunk_id_map[edge["end_node_id"]],
            }
        )
print("출처에 연결할 개체:", len(paper_stored_entity_ids), "/ 연결할 기록:", len(paper_source_links))

#### 표준 개체 ID와 고정 청크 ID로 연결

In [ ]:
paper_source_query = """
UNWIND $rows AS item
MATCH (e:$(item.entity_type) {standard_id: item.standard_id})
MATCH (c:Chunk {chunk_id: item.chunk_id})
MERGE (e)-[:FROM_CHUNK]->(c)
RETURN count(*) AS processed // 기존 연결을 재사용한 기록도 셉니다.
"""
paper_source_result = run_cypher(paper_source_query, rows=paper_source_links)
print("처리한 출처 연결:", paper_source_result[0]["processed"])

#### 출처 문서와 원문 조회

In [ ]:
paper_source_chunk_ids = [row["chunk_id"] for row in paper_chunk_rows]
paper_read_sources_query = """
MATCH (e)-[link:FROM_CHUNK]->(c:Chunk)-[parent:FROM_DOCUMENT]->(d:Document)
WHERE c.chunk_id IN $chunk_ids AND e.standard_id IN $entity_ids
RETURN e.standard_id AS standard_id, // 개체 ID입니다.
       d.doc_id AS doc_id, d.title AS title, d.url AS url, // 출처 문서입니다.
       c.chunk_id AS chunk_id, c.index AS index, c.text AS text, // 출처 청크입니다.
       size(c.embedding) AS dimensions, // 청크의 임베딩 차원입니다.
       elementId(d) AS document_node, elementId(c) AS chunk_node, // 재사용 여부를 비교할 DB ID입니다.
       elementId(link) AS source_link, elementId(parent) AS document_link // 출처 연결도 중복되지 않는지 비교합니다.
ORDER BY standard_id, chunk_id
"""
paper_sources_after = run_cypher(
    paper_read_sources_query,
    chunk_ids=paper_source_chunk_ids,
    entity_ids=list(paper_stored_entity_ids),
)
for source in paper_sources_after:
    print("개체:", source["standard_id"], "/ 문서:", source["doc_id"])
    print("청크 원문:", source["text"])
    print()

#### 3-3. 문서, 청크와 관계를 같은 입력으로 재실행

In [ ]:
# 같은 키로 문서, 청크와 출처 연결을 다시 저장합니다.
run_cypher(paper_document_query, rows=paper_document_rows)
run_cypher(paper_chunk_query, rows=paper_chunk_rows)
run_cypher(paper_source_query, rows=paper_source_links)
paper_sources_replayed = run_cypher(
    paper_read_sources_query,
    chunk_ids=paper_source_chunk_ids,
    entity_ids=list(paper_stored_entity_ids),
)
print("문서, 청크와 출처 연결의 ID 및 내용 유지:", paper_sources_after == paper_sources_replayed)

#### 전체 문서와 청크 수 확인

In [ ]:
# 개체와 연결되지 않은 청크도 포함해 저장한 전체 문서와 청크를 셉니다.
paper_saved_counts = run_cypher(
    """
MATCH (d:Document)
WHERE d.doc_id IN $doc_ids
OPTIONAL MATCH (c:Chunk)-[:FROM_DOCUMENT]->(d)
WHERE c.chunk_id IN $chunk_ids
RETURN count(DISTINCT d) AS documents, // 문서 노드 수입니다.
       count(DISTINCT c) AS chunks // 이번에 저장한 청크 노드 수입니다.
""",
    doc_ids=[row["doc_id"] for row in paper_document_rows],
    chunk_ids=paper_source_chunk_ids,
)[0]
print("재실행 후 문서:", paper_saved_counts["documents"], "/ 청크:", paper_saved_counts["chunks"])

#### 추출 관계도 재실행 확인

In [ ]:
paper_replay_processed = put_claims(paper_ready, paper_batch_id, paper_schema_version)
paper_replayed = read_claims(paper_document_ids, paper_known_ids)
print("다시 처리한 행:", paper_replay_processed)
print("재실행 전후 관계 수:", len(paper_after), "/", len(paper_replayed))
print("ID와 근거까지 동일:", paper_after == paper_replayed)

### 4. 서로 다른 개념을 계층으로 연결해 조회합니다

#### 4-1. 계층 연결 함수 준비

#### 상위 개념으로 조회 범위 확장하기

기존 치료 관계는 유지하고 질환 계층으로 찾습니다.

In [ ]:
def put_hierarchy(edges):
    """하위 개념에서 상위 개념으로 향하는 BROADER 관계를 만듭니다.

    Args:
        edges: child_id, parent_id와 상위 개념의 이름, 타입, 정의 이유를 담은 목록.
    Returns:
        처리한 계층 관계 수. 이미 있던 관계도 포함합니다.
    """
    # 상위 개념 노드가 없으면 먼저 만듭니다.
    run_cypher(
        """
    UNWIND $edges AS item
    MERGE (parent:$(item.parent_type) {standard_id: item.parent_id})
    ON CREATE SET parent.name = item.parent_name
    RETURN count(parent) AS processed // 이미 있던 개념도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    # 상위 개념을 모두 준비한 뒤 하위 개념과 연결합니다.
    rows = run_cypher(
        """
    UNWIND $edges AS item
    // ID로 하위 개념과 상위 개념 노드를 찾습니다.
    MATCH (child {standard_id: item.child_id})
    MATCH (parent {standard_id: item.parent_id})
    // 원문에서 추출한 관계와 구분해, 개념의 상하 관계는 BROADER로 연결합니다.
    MERGE (child)-[r:BROADER]->(parent)
    ON CREATE SET r.note = item.note // 왜 이 계층을 정했는지 남깁니다.
    RETURN count(r) AS processed // 이미 있던 관계도 처리 수에 포함합니다.
    """,
        edges=edges,
    )
    return rows[0]["processed"]

#### 4-2. 계층 전후 같은 질문으로 조회

In [ ]:
paper_top_id = "Disease::DOID:8893"
paper_broad_query = """
MATCH (c:Compound)-[r:TREATS]->(d:Disease)-[:BROADER*0..]->(:Disease {standard_id: $top_id})
WHERE r.claim_id IS NOT NULL AND c.standard_id IN $known_ids
RETURN DISTINCT c.name AS compound, d.name AS disease // 원래 치료 관계의 약물과 질환입니다.
ORDER BY compound, disease
"""
paper_before_rows = run_cypher(
    paper_broad_query, top_id=paper_top_id, known_ids=paper_known_ids
)
print("계층 연결 전:", paper_before_rows)


# paper_hierarchy.json: 원문 추출과 별도로 정한 개념의 상하 관계입니다.
paper_hierarchy = read_json("paper_hierarchy.json")
paper_hierarchy_processed = put_hierarchy(paper_hierarchy)
paper_after_rows = run_cypher(
    paper_broad_query, top_id=paper_top_id, known_ids=paper_known_ids
)
print("처리한 계층 관계:", paper_hierarchy_processed)
print("계층 연결 후:", paper_after_rows)

### 5. 처리 이력을 남기고 이번 추가만 되돌립니다

#### 5-1. 문서별 이력 기록

#### 적재 이력 함수

실행한 문서의 버전과 검사 건수를 남깁니다.

In [ ]:
def record_state(document, version, ready, pending, rejected, batch_id):
    """문서별 마지막 적재 처리의 버전, 검사 건수와 상태를 기록합니다.

    Args:
        document: 출처 doc_id가 있는 원문.
        version: 검사에 적용한 규칙 버전.
        ready, pending, rejected: 해당 문서의 검사 결과 목록.
        batch_id: 이번 적재 묶음 ID.
    Returns:
        문서 ID, 버전, 적재 상태와 검사 건수의 딕셔너리.
    """
    rows = run_cypher(
        """
    MERGE (d:ProcessingState {doc_id: $doc_id})
    SET d.schema_version = $version, d.status = 'stored', d.batch_id = $batch_id,
        d.accepted = $accepted, d.pending = $pending, d.rejected = $rejected,
        d.processed_at = datetime()
    RETURN d.doc_id AS doc_id, d.schema_version AS schema_version, // 문서와 적용 규칙입니다.
           d.status AS status, // stored는 적재 가능 행의 저장을 마친 상태입니다.
           d.accepted AS accepted, d.pending AS pending, d.rejected AS rejected // 이번 검사 건수입니다.
    """,
        doc_id=document["doc_id"],
        version=version,
        batch_id=batch_id,
        accepted=len(ready),
        pending=len(pending),
        rejected=len(rejected),
    )
    return rows[0]

#### 의료 적재 이력 저장

In [ ]:
run_cypher(
    "CREATE CONSTRAINT IF NOT EXISTS FOR (d:ProcessingState) REQUIRE d.doc_id IS UNIQUE"
)
paper_storage_states = []
# 관계가 0건이어도 추출을 완료한 문서는 처리 이력에 남깁니다.
for run in paper_packet["builder_runs"]:
    document = run["document"]
    doc_id = document["doc_id"]
    ready_rows = [row for row in paper_ready if row["source_doc_id"] == doc_id]
    pending_rows = [row for row in paper_pending if row["source_doc_id"] == doc_id]
    rejected_rows = [row for row in paper_rejected if row["source_doc_id"] == doc_id]
    state = record_state(
        document,
        paper_schema_version,
        ready_rows,
        pending_rows,
        rejected_rows,
        paper_batch_id,
    )
    paper_storage_states.append(state)
    pprint(state)

#### 5-2. 이번 추가만 되돌리고 연결 종료

In [ ]:
# 이번 batch_id로 처음 만든 관계만 지웁니다. 이전 관계, 노드와 BROADER는 유지합니다.
paper_removed = run_cypher(
    """
MATCH ()-[r]->()
WHERE r.batch_id = $batch_id
DELETE r
RETURN count(*) AS removed // 이번 묶음에서 삭제한 관계 수입니다.
""",
    batch_id=paper_batch_id,
)[0]
paper_restored = read_claims(paper_document_ids, paper_known_ids)
print("되돌린 관계 수:", paper_removed["removed"])
print("남은 기존 관계:", len(paper_restored))
print("기존 관계와 근거 복원:", paper_restored == paper_before)

# 관계가 0건인 문서의 완료 상태도 이번 묶음과 함께 되돌립니다.
paper_reverted_states = run_cypher(
    """
MATCH (d:ProcessingState {batch_id: $batch_id})
SET d.status = 'reverted'
RETURN d.doc_id AS doc_id, d.status AS status // 추가 적재를 되돌린 문서입니다.
ORDER BY doc_id
""",
    batch_id=paper_batch_id,
)
pprint(paper_reverted_states)


# 추출 관계를 되돌려도 문서, 청크와 개체의 출처 연결은 남깁니다.
paper_sources_restored = run_cypher(
    paper_read_sources_query,
    chunk_ids=paper_source_chunk_ids,
    entity_ids=list(paper_stored_entity_ids),
)
print("되돌린 후에도 출처 유지:", paper_sources_after == paper_sources_restored)

driver.close()